# Portion 3 · Fine-tuning grid

Runs the 20 remaining fine-tuned encoder runs (Experiment A plus the Experiment C
ablations) by calling Portion 2's `finetune()` unchanged. Reference run
`ft_main_seed42_5pct_seed42` already exists on the shared Drive folder and is skipped.

- GPU required: RTX 5080 (16 GB) or a Colab T4. About 2 to 2.5 hours for the grid.
- Outputs go to `<out>/finetune/`; finished runs are skipped, so a disconnect only loses
  the run in progress.
- On Colab, outputs go to Drive. On the PC, point `TABBERT_OUT` at the shared folder or a
  local copy of it.

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    REPO = Path("/content/TabBERT_AML")
    if not (REPO / "Portion3").exists():
        subprocess.run(["git", "clone", "--branch", "portion3",
                        "https://github.com/ted-cc6/TabBERT_AML.git", str(REPO)], check=True)
        subprocess.run(["git", "lfs", "install"], cwd=REPO, check=True)
        subprocess.run(["git", "lfs", "pull"], cwd=REPO, check=True)
else:
    REPO = Path.cwd()
    if not (REPO / "Portion3").exists():
        REPO = REPO.parent
sys.path.insert(0, str(REPO))
print("repo:", REPO)

## 1. Check the GPU

The RTX 5080 is Blackwell (compute capability 12.0) and needs a CUDA 12.8 PyTorch build.
If `torch.cuda.is_available()` is False on the PC:
`pip install torch --index-url https://download.pytorch.org/whl/cu128`
On Colab, pick an L4 or T4 runtime; the image already has a matching build.

In [ ]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "| memory GB:",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 2. Locate the checkpoints and score outputs

The checkpoint folder is the shared `portion2_runs` folder (Drive on Colab; a local copy or
the Drive-synced folder on the PC). Set `TABBERT_OUT` to override the PC path.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/TabBERT_AML/portion2_runs")
else:
    OUT_ROOT = Path(os.environ.get("TABBERT_OUT", REPO / "Portion3" / "outputs" / "portion2_runs"))
for run in ("main_seed42", "main_seed43", "main_seed44", "abl_whole_mask_seed42", "abl_no_dt_seed42"):
    print(run, (OUT_ROOT / run / "checkpoint.pt").exists())

## 3. Run the grid

20 runs: 1/2/5/20/100% labels x 3 seeds, plus the two ablations at 5% x 3 seeds.
Add `--max-epochs 3` for a quick end-to-end check, or `--runs <id> ...` for a subset.

In [ ]:
cmd = [sys.executable, str(REPO / "Portion3" / "run_finetune.py"),
       "--checkpoint-dir", str(OUT_ROOT), "--out-dir", str(OUT_ROOT)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 4. Score the runs into the shared log

`score-finetune` applies the same metrics module as every tabular run, so all methods are
comparable. On Colab the clone's `results/runs.csv` is ephemeral: copy `<out>/finetune`
back to the main machine and run the command there instead.

In [ ]:
if IN_COLAB:
    print("On Colab, run this after copying the finetune folder to the main machine:")
    print(f"python -m Portion3.portion3_baselines score-finetune --scores-root {OUT_ROOT / 'finetune'}")
else:
    subprocess.run([sys.executable, "-m", "Portion3.portion3_baselines", "score-finetune",
                    "--scores-root", str(OUT_ROOT / "finetune")], cwd=REPO, check=True)

## 5. Summary table

In [ ]:
import pandas as pd
results = REPO / "results" / "runs.csv"
if results.exists():
    frame = pd.read_csv(results)
    pd.set_option("display.width", 200)
    print(frame[["run_id", "method", "label_fraction", "seed", "valid_ap", "test_ap", "test_f1"]].to_string(index=False))